# FedAS Server

>

In [ ]:
#| default_exp servers.fedas

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import gc
import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs


In [ ]:
#| export
@AlgorithmRegistry.register_server("fedas")
class ServerFedAS(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
        

In [ ]:
#| export
@patch
def client_fn(self: ServerFedAS, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = self.model.state_dict()
        client_state['fim_trace_history'] = []

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model
    
    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    
    optimizer = get_optimizer(self.cfg)(client_state['model'].parameters(), **kwargs)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)
    client_state['optimizer'] = optimizer


    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


In [ ]:
#| export
@patch
def train(self: ServerFedAS):

    res =  []
    selected_clients = self.client_selector.select()
    for t in range(1, self.cfg.n_rounds + 1):
        self.lst_active_ids = selected_clients[t-1]
        self.alled_clients = list(range(self.cfg.num_clients))
        len_clients_ds = {}

        for id in self.alled_clients:
            client_state = self.state_mgr.get_state(id)
            client = self.client_fn(id= id, comm_round= t, client_state= client_state)
            client.set_parameters(self.model)
            client.fit(client.id in self.lst_active_ids)
            
            self.logger.info(f"Client {id} finished training.")
            self.logger.info("*"*20)
            self.state_mgr.set_state(id, client.save_state())
            
            len_clients_ds[id] = len(client.train_loader.dataset)

            del client 
            gc.collect()
            torch.cuda.empty_cache()

        self.aggregate(self.lst_active_ids, t, len_clients_ds)#lst_active_ids, comm_round, len_clients_ds
        train_res, test_res = self.evaluate(t)
        
        train_df, test_df = self.writer.write(self.lst_active_ids, train_res, test_res, t) 
        res.append((train_df, test_df))
        
    self.writer.save(res)
    self.writer.finish()

    return res

### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerFedAS, lst_active_ids, comm_round, len_clients_ds):


    FIM_weight_list = []
    for id in lst_active_ids:
        client_fim = self.state_mgr.get_state(id).get('fim_trace_history', None)
        FIM_weight_list.append(client_fim[-1])
    FIM_weight_list = [FIM_value/sum(FIM_weight_list) for FIM_value in FIM_weight_list]

    global_model = None
    
    for id in lst_active_ids:
        client_state_dict = self.state_mgr.get_state(id).get('model', None)

        if global_model is None:
            global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

        for key in global_model.keys():
            global_model[key].add_(client_state_dict[key], alpha=FIM_weight_list[lst_active_ids.index(id)])

    self.model.load_state_dict(global_model)
    
    for id in lst_active_ids:
        client_model = self.state_mgr.get_state(id).get('model', None)
        for key in global_model.keys():
            client_model[key] = self.model.state_dict()[key]

        self.state_mgr.set_state(id, {
                **self.state_mgr.get_state(id),
                'model': client_model,
            })

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()